# Evaluate SAC HER Model

This notebook evaluates a trained SAC HER model on the Flycraft environment.

In [20]:
import sys
from pathlib import Path
import argparse

# Add project root to path
PROJECT_ROOT_DIR = Path().absolute().parent
if str(PROJECT_ROOT_DIR.absolute()) not in sys.path:
    sys.path.append(str(PROJECT_ROOT_DIR.absolute()))

from stable_baselines3 import SAC
from utils_my.sb3.vec_env_helper import get_vec_env
from utils_my.sb3.my_evaluate_policy import evaluate_policy_with_success_rate

In [21]:
# Configuration
# Update these paths to point to your specific config and model
CONFIG_FILE_NAME = "configs/train/sac_her_default.json"
MODEL_PATH = "checkpoints/sac_her/best_model/best_model.zip"

EVAL_EPISODES = 50
SEED = 1
DETERMINISTIC = True

config_path = PROJECT_ROOT_DIR / CONFIG_FILE_NAME
model_path = PROJECT_ROOT_DIR / MODEL_PATH

print(f"Config: {config_path}")
print(f"Model: {model_path}")

Config: /home/hs/dev_ai/codes/IRPO/exp_on_flycraft/configs/train/sac_her_default.json
Model: /home/hs/dev_ai/codes/IRPO/exp_on_flycraft/checkpoints/sac_her/best_model/best_model.zip


In [22]:
# Initialize Environment
from flycraft.utils_common.load_config import load_config

# Load training config to get the environment config path
train_config = load_config(config_path)
env_config_file_name = train_config["env"]["config_file"]
env_config_path = PROJECT_ROOT_DIR / "configs" / "env" / env_config_file_name

print(f"Loading env config from: {env_config_path}")

# Using get_vec_env to ensure same wrappers (ScaledObservationWrapper, ScaledActionWrapper) as training
env = get_vec_env(
    num_process=1, 
    seed=SEED, 
    config_file=env_config_path,
    custom_config={"debug_mode": False, "flag_str": "Evaluate"}
)

print("Environment initialized.")

Loading env config from: /home/hs/dev_ai/codes/IRPO/exp_on_flycraft/configs/env/env_config_for_ppo.json
load config from: /home/hs/dev_ai/codes/IRPO/exp_on_flycraft/configs/env/env_config_for_ppo.json
1 Generator(PCG64) Generator(PCG64)
Environment initialized.


In [23]:
# Load Model
# Using custom_objects to ensure compatibility if environments differ slightly in structure (e.g. wrapper references)
model = SAC.load(
    model_path, 
    env=env,
    custom_objects={
        "observation_space": env.observation_space,
        "action_space": env.action_space
    }
)

print("Model loaded successfully.")

verbose:  0
Model loaded successfully.


In [24]:
# Run Evaluation
print(f"Evaluating for {EVAL_EPISODES} episodes...")

mean_reward, std_reward, success_rate = evaluate_policy_with_success_rate(
    model=model,
    env=env,
    n_eval_episodes=EVAL_EPISODES,
    deterministic=DETERMINISTIC
)

print("-" * 50)
print(f"Mean Reward: {mean_reward:.2f} +/- {std_reward:.2f}")
print(f"Success Rate: {success_rate * 100:.2f}%")
print("-" * 50)

Evaluating for 50 episodes...


/home/hs/dev_ai/codes/IRPO/exp_on_flycraft/utils_my/sb3/my_evaluate_policy.py:67: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


--------------------------------------------------
Mean Reward: -160.83 +/- 52.63
Success Rate: 16.00%
--------------------------------------------------
